# Stock Prices - Exploratory Data Analysis

This notebook performs comprehensive exploratory data analysis on the stock prices dataset, including:
- Missing values analysis
- Outliers detection and documentation
- Price trend visualizations
- Data quality assessment

## 1. Load Data

In [27]:
import pandas as pd
import os
import matplotlib.pyplot as plt
import os
prices_df = pd.read_csv('../data/stock_prices.csv')

prices_df['Date'] = pd.to_datetime(prices_df['Date'])

## 2. Missing Values Analysis

In [28]:


os.makedirs('results/plots', exist_ok=True)

prices_df['Date'] = pd.to_datetime(prices_df['Date'])
prices_df['year'] = prices_df['Date'].dt.year

ticker_columns = [col for col in prices_df.columns if col not in ['Date', 'year']]

missing_per_year = prices_df.groupby('year')[ticker_columns].apply(lambda x: x.isnull().sum().sum())

# Create bar chart
plt.figure(figsize=(12, 6))
plt.bar(missing_per_year.index.astype(str), missing_per_year.values, color='coral', edgecolor='black')
plt.title('Missing Values per Year - All Tickers Combined', fontsize=14)
plt.xlabel('Year', fontsize=12)
plt.ylabel('Number of Missing Values', fontsize=12)
plt.xticks(rotation=45)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('../results/plots/missing_per_year.png', dpi=300, bbox_inches='tight')
plt.close()


## 3. Outliers Analysis

In [22]:


os.makedirs('results/plots', exist_ok=True)

# Your outlier detection code
outlier_rows = []

for col in prices_df.columns[1:]:
    Q1 = prices_df[col].quantile(0.25)
    Q3 = prices_df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    mask = (prices_df[col] < lower) | (prices_df[col] > upper)
    
    if mask.any():
        idx = mask.idxmax()
        outlier_rows.append({
            'date': prices_df.loc[idx, 'Date'],
            'ticker': col,
            'price': prices_df.loc[idx, col]
        })

outliers = pd.DataFrame(outlier_rows).sort_values("date").reset_index(drop=True)
print(outliers)

# Graph: Number of outliers per ticker
plt.figure(figsize=(12, 6))
outlier_counts = outliers['ticker'].value_counts().head(20)
plt.barh(outlier_counts.index, outlier_counts.values, color='coral', edgecolor='black')
plt.title('Number of Outliers per Ticker', fontsize=14)
plt.xlabel('Number of Outliers', fontsize=12)
plt.ylabel('Ticker', fontsize=12)
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('../results/plots/outliers_per_ticker.png', dpi=300, bbox_inches='tight')
plt.close()

print("\n✅ Outlier plot saved: results/plots/outliers_per_ticker.png")

          date ticker       price
0   2000-12-12    THC  105.333333
1   2000-12-12    RSH   42.277341
2   2000-12-12   BRCM   88.586549
3   2000-12-12   NTAP   80.741692
4   2000-12-12     MU   36.250000
..         ...    ...         ...
297 2014-12-11    HSP   62.080000
298 2014-12-18    ALL   69.790000
299 2014-12-18    IRM   40.270000
300 2014-12-22    AIV   38.320000
301 2014-12-26     NU   55.670000

[302 rows x 3 columns]

✅ Outlier plot saved: results/plots/outliers_per_ticker.png


## 4. Price Trends Over Time

In [29]:


def plot_average_price_over_time(plot=False):
    os.makedirs('results/plots', exist_ok=True)
    
    # Calculate average price across all companies for each date
    avg_price_by_date = prices_df.iloc[:, 1:].mean(axis=1, skipna=True)
    
    plt.figure(figsize=(14, 6))
    plt.plot(prices_df['Date'], avg_price_by_date, linewidth=2, color='steelblue')
    plt.title('Average Price Across All Companies Over Time', fontsize=14)
    plt.xlabel('Date', fontsize=12)
    plt.ylabel('Average Price ($)', fontsize=12)
    plt.xticks(rotation=45)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    
    if plot:
        plt.savefig('../results/plots/average_price_over_time.png', dpi=300, bbox_inches='tight')
    plt.close()

plot_average_price_over_time(plot=True)

In [33]:
outliers_df = pd.DataFrame(outlier_rows, columns=['date', 'ticker', 'price']).sort_values('price', ascending=False)

combined_df = pd.concat([outliers_df.head(3), outliers_df.tail(2)])

print(combined_df)

combined_df.sort_values('date').to_csv('../results/outliers.txt', sep='\t', index=False)

          date ticker          price
242 2000-12-15   SPLS   2.346450e+07
210 2013-11-26   PCLN   1.177980e+03
269 2000-12-12    TYC   1.082437e+03
67  2014-04-03    CSC  1.399589e-317
260 2014-03-24    TMK  6.392715e-318
